## 코드 받아오기

Colab은 매번 빈 서버라서, GitHub에서 코드를 통째로 내려받아야 한다.
두 번째부터는 `already exists` 에러가 나는데 무시해도 된다.

In [ ]:
!git clone https://github.com/chowin131/temp.git
%cd /content/temp

In [ ]:
import torch

import main

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch  :", torch.__version__)
print("device :", device)

In [ ]:
import architectures
import datasets
import methods

print("architectures:", list(architectures.ARCHITECTURES_REGISTRY))
print("method :", list(methods.METHOD_REGISTRY))
print("dataset:", list(datasets.NUM_CLASSES))
print()
for name, blocks in architectures.DEFAULT_BLOCKS.items():
    print(f"  {name:16s} 기본 blocks = {blocks}")

In [ ]:
import torch.optim as optim

import trainer

train_dataloader, test_dataloader = datasets.get_dataloaders(
    "cifar10", batch_size=128, num_workers=2,
)

encoder = architectures.get_encoder("resnet", [3, 3, 3])
method = methods.get_method("supervised", encoder, num_classes=10).to(device)
optimizer = optim.SGD(method.parameters(), lr=0.1, momentum=0.9,
                      weight_decay=1e-4)

print(encoder.__class__.__name__, "->", method.classifier)

batch = trainer.to_device(next(iter(train_dataloader)), device)
loss = method(batch)
loss.backward()
optimizer.step()
print("loss =", loss.item())

### FractalNet

In [ ]:
# 재귀 버전, C=3인 block 5개 -> 20 layer
result, best_acc = main.run_experiment(
    arch="fractalnet", blocks=[3, 3, 3, 3, 3],
    arch_kwargs={"p_local": 0.15, "p_global": 0.5},
    run_name="fractalnet_c3",
)

### ViT

In [ ]:
result, best_acc = main.run_experiment(
    arch="vit", blocks=[12],
    arch_kwargs={"patch_size": 4, "latent_vector_size": 192,
                 "num_heads": 3, "p_drop": 0.1},
    lr=0.01, weight_decay=5e-5, run_name="vit_ti4",
)

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
import utils

runs = {}
for path in sorted(glob.glob("runs/*.json")):
    name = os.path.splitext(os.path.basename(path))[0]
    runs[name] = utils.load_result(path)

print(f"{len(runs)}개 run :", list(runs))

In [ ]:
# 요약 표
print(f"{'run':28s} {'epochs':>7s} {'best acc':>9s} {'final acc':>10s}")
print("-" * 58)
for name, r in sorted(runs.items(), key=lambda kv: -max(kv[1]["test_acc"])):
    print(f"{name:28s} {len(r['test_acc']):7d} "
          f"{max(r['test_acc']) * 100:8.2f}% {r['test_acc'][-1] * 100:9.2f}%")

In [ ]:
# run 하나 자세히 보기 (loss / test acc / lr)
name = list(runs)[0]
utils.plot_result(runs[name], title=name)
plt.show()

In [ ]:
# 여러 run 겹쳐 그리기
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for name, r in sorted(runs.items()):
    epochs = range(1, len(r["train_loss"]) + 1)
    axes[0].plot(epochs, r["train_loss"], label=name)
    axes[1].plot(epochs, [a * 100 for a in r["test_acc"]], label=name)

axes[0].set_xlabel("epoch")
axes[0].set_ylabel("train loss")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("test acc (%)")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
plt.show()